In [ ]:
# CELL 1 — SETTINGS, DATASET DISCOVERY, AND SQL ENGINE
import sys, subprocess, importlib.util
if importlib.util.find_spec('duckdb') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'duckdb>=1.1,<2'])

import duckdb, pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

DATA_FILE = None  # e.g. '/content/Motor_Vehicle_Collisions_-_Crashes.csv'
ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
OUT = ROOT / 'NYC_Project7_Outputs'
OUT.mkdir(exist_ok=True, parents=True)

if DATA_FILE is None:
    candidates = sorted([f for f in ROOT.glob('*.csv') if f.is_file()], key=lambda x: x.stat().st_size, reverse=True)
    # Ignore CSV summaries from previous executions.
    candidates = [f for f in candidates if not f.name.startswith('nyc_')]
    print('CSV files found:', [f.name for f in candidates])
    assert candidates, 'Upload the NYC Motor Vehicle Collisions CSV to Colab or set DATA_FILE.'
    DATA_FILE = str(candidates[0])
DATA_FILE = Path(DATA_FILE)
assert DATA_FILE.exists(), f'File not found: {DATA_FILE}'
print('Selected data file:', DATA_FILE, 'Size (MB):', round(DATA_FILE.stat().st_size/1e6,1))

# Persistent SQL database (not included in final ZIP due to size).
DB_FILE = OUT / 'nyc_collisions.duckdb'
con = duckdb.connect(str(DB_FILE))
con.execute("SET memory_limit='3GB'")
con.execute('SET threads=2')


In [ ]:
# CELL 2 — LOAD INTO SQL AND MAP REAL DATASET COLUMNS
# Ingest as strings to prevent inference errors on millions of rows.
# DuckDB reads the CSV with a streaming SQL scanner; no full pandas load.
src = str(DATA_FILE).replace("'", "''")
con.execute(f"CREATE OR REPLACE TABLE source_crashes AS SELECT * FROM read_csv('{src}', header=true, all_varchar=true, null_padding=true, strict_mode=false, ignore_errors=true)")
original_cols = con.execute('DESCRIBE source_crashes').df()['column_name'].tolist()
print('Source columns:', len(original_cols))
print(original_cols)

def norm(s):
    return ''.join(ch if ch.isalnum() else '_' for ch in str(s).strip().lower()).strip('_')
colmap = {norm(c): c for c in original_cols}
def field(*aliases):
    for a in aliases:
        if norm(a) in colmap:
            return '"' + colmap[norm(a)].replace('"','""') + '"'
    return 'NULL'

required = {'crash_date': field('CRASH DATE','CRASH_DATE','collision_date')}
assert required['crash_date'] != 'NULL', 'Could not find CRASH DATE. Inspect printed columns and rename source field.'

fields = {
    'crash_time':field('CRASH TIME','CRASH_TIME'),
    'borough':field('BOROUGH'), 'on_street':field('ON STREET NAME','ON_STREET_NAME'),
    'cross_street':field('CROSS STREET NAME','CROSS_STREET_NAME'),
    'off_street':field('OFF STREET NAME','OFF_STREET_NAME'),
    'latitude':field('LATITUDE'), 'longitude':field('LONGITUDE'),
    'collision_id':field('COLLISION_ID','COLLISION ID'),
    'injured':field('NUMBER OF PERSONS INJURED'),
    'killed':field('NUMBER OF PERSONS KILLED'),
    'ped_inj':field('NUMBER OF PEDESTRIANS INJURED'),
    'ped_kill':field('NUMBER OF PEDESTRIANS KILLED'),
    'cyc_inj':field('NUMBER OF CYCLIST INJURED','NUMBER OF CYCLISTS INJURED'),
    'cyc_kill':field('NUMBER OF CYCLIST KILLED','NUMBER OF CYCLISTS KILLED'),
    'mot_inj':field('NUMBER OF MOTORIST INJURED','NUMBER OF MOTORISTS INJURED'),
    'mot_kill':field('NUMBER OF MOTORIST KILLED','NUMBER OF MOTORISTS KILLED'),
}
factor_fields = [field(f'CONTRIBUTING FACTOR VEHICLE {i}') for i in range(1,6)]
vehicle_fields = [field(f'VEHICLE TYPE CODE {i}') for i in range(1,6)]
print('Mapped fields:')
display(pd.DataFrame({'field':list(fields),'source':[x if x != 'NULL' else 'MISSING' for x in fields.values()]}))
assert any(x != 'NULL' for x in factor_fields), 'No contributing factor columns found.'
assert any(x != 'NULL' for x in vehicle_fields), 'No vehicle type columns found.'

def num(expr): return f"TRY_CAST(NULLIF(TRIM({expr}), '') AS DOUBLE)" if expr != 'NULL' else 'NULL::DOUBLE'
def txt(expr): return f"NULLIF(TRIM({expr}), '')" if expr != 'NULL' else 'NULL::VARCHAR'
date_expr = "COALESCE(" + ", ".join([f"TRY_STRPTIME({required['crash_date']}, '{fmt}')" for fmt in ['%m/%d/%Y','%Y-%m-%d','%m/%d/%y','%Y/%m/%d']]) + ')'
time_raw = txt(fields['crash_time'])
time_expr = ("COALESCE(TRY_STRPTIME(" + time_raw + ", '%H:%M'), TRY_STRPTIME(" + time_raw + ", '%H:%M:%S'), TRY_STRPTIME(" + time_raw + ", '%I:%M %p'))::TIME") if fields['crash_time'] != 'NULL' else 'NULL::TIME'
selected = [
    f'{date_expr}::DATE AS crash_date', f'{time_expr} AS crash_time',
    f"UPPER(COALESCE({txt(fields['borough'])}, 'UNKNOWN')) AS borough",
    f"UPPER({txt(fields['on_street'])}) AS on_street",
    f"UPPER({txt(fields['cross_street'])}) AS cross_street",
    f"UPPER({txt(fields['off_street'])}) AS off_street",
    f'{num(fields["latitude"])} AS latitude', f'{num(fields["longitude"])} AS longitude',
    f'{txt(fields["collision_id"])} AS collision_id',
]
for k in ['injured','killed','ped_inj','ped_kill','cyc_inj','cyc_kill','mot_inj','mot_kill']:
    selected.append(f'{num(fields[k])} AS {k}')
for i,x in enumerate(factor_fields,1): selected.append(f'UPPER({txt(x)}) AS factor_{i}')
for i,x in enumerate(vehicle_fields,1): selected.append(f'UPPER({txt(x)}) AS vehicle_{i}')
con.execute('CREATE OR REPLACE TABLE crashes AS SELECT ' + ', '.join(selected) + ' FROM source_crashes')
print('Loaded rows:', con.execute('SELECT COUNT(*) FROM crashes').fetchone()[0])


In [ ]:
# CELL 3 — AUDIT AND CLEAN STANDARDIZED CRASH, FACTOR, VEHICLE VIEWS
# Counts are collision-record counts. Rate comparisons need exposure data
# (traffic volume/population), which this crash-only source does not provide.
con.execute("""
CREATE OR REPLACE VIEW valid_crashes AS
SELECT *, YEAR(crash_date)::INTEGER AS year,
       DATE_TRUNC('month',crash_date)::DATE AS month,
       COALESCE(injured,0) + COALESCE(killed,0) AS casualties,
       CASE WHEN COALESCE(injured,0)+COALESCE(killed,0)>0 THEN 1 ELSE 0 END AS casualty_crash
FROM crashes WHERE crash_date IS NOT NULL
""")
con.execute("""
CREATE OR REPLACE VIEW crash_factors AS
SELECT collision_id, crash_date, year, borough, factor
FROM valid_crashes UNPIVOT (factor FOR factor_slot IN (factor_1,factor_2,factor_3,factor_4,factor_5))
WHERE factor IS NOT NULL AND factor NOT IN ('UNSPECIFIED','UNKNOWN','N/A','OTHER')
""")
con.execute("""
CREATE OR REPLACE VIEW crash_vehicles AS
SELECT collision_id, crash_date, year, borough, casualty_crash, injured, killed,
CASE
 WHEN REGEXP_MATCHES(vehicle, 'SPORT UTILITY|STATION WAGON|SUV') THEN 'SUV / STATION WAGON'
 WHEN REGEXP_MATCHES(vehicle, 'SEDAN|4 DR SEDAN|PASSENGER VEHICLE') THEN 'SEDAN / PASSENGER CAR'
 WHEN REGEXP_MATCHES(vehicle, 'PICK.UP|TRUCK|VAN|TRACTOR') THEN 'TRUCK / VAN'
 WHEN REGEXP_MATCHES(vehicle, 'BICYCLE|BIKE|E-BIKE|E BIKE') THEN 'BICYCLE / E-BIKE'
 WHEN REGEXP_MATCHES(vehicle, 'MOPED|MOTORCYCLE|SCOOTER') THEN 'MOTORCYCLE / MOPED'
 WHEN REGEXP_MATCHES(vehicle, 'BUS') THEN 'BUS'
 WHEN REGEXP_MATCHES(vehicle, 'TAXI') THEN 'TAXI'
 ELSE vehicle END AS vehicle_group
FROM valid_crashes UNPIVOT (vehicle FOR vehicle_slot IN (vehicle_1,vehicle_2,vehicle_3,vehicle_4,vehicle_5))
WHERE vehicle IS NOT NULL AND vehicle NOT IN ('UNSPECIFIED','UNKNOWN','N/A')
""")

audit = con.execute("""
SELECT COUNT(*) source_records,
 COUNT(*) FILTER (WHERE crash_date IS NULL) missing_or_bad_dates,
 COUNT(*) FILTER (WHERE borough='UNKNOWN') missing_borough,
 COUNT(*) FILTER (WHERE on_street IS NULL AND cross_street IS NULL AND off_street IS NULL) missing_all_streets,
 COUNT(*) FILTER (WHERE latitude IS NULL OR longitude IS NULL) missing_coordinates,
 COUNT(*) FILTER (WHERE collision_id IS NULL) missing_collision_ids,
 COUNT(*) - COUNT(DISTINCT collision_id) AS extra_rows_beyond_distinct_ids_including_nulls,
 COUNT(*) FILTER (WHERE injured IS NULL OR killed IS NULL) missing_person_total_fields,
 COUNT(*) FILTER (WHERE injured < 0 OR killed < 0) negative_casualty_records,
 MIN(crash_date) earliest_date, MAX(crash_date) latest_date
FROM crashes
""").df()
display(audit.T.rename(columns={0:'value'}))
audit.to_csv(OUT/'nyc_data_quality_audit.csv',index=False)

# Document collisions with repeated IDs separately. Do not quietly discard source rows.
duplicate_ids=con.execute("""
SELECT collision_id, COUNT(*) AS occurrences FROM crashes
WHERE collision_id IS NOT NULL GROUP BY collision_id HAVING COUNT(*)>1 ORDER BY occurrences DESC
""").df()
duplicate_ids.to_csv(OUT/'nyc_duplicate_collision_ids_audit.csv',index=False)
print('Repeated collision identifiers:',len(duplicate_ids))
print('Distinct years with valid dates:',con.execute('SELECT COUNT(DISTINCT year) FROM valid_crashes').fetchone()[0])


In [ ]:
# CELL 4 — SQL COLLISION TRENDS, BOROUGHS, STREETS, CAUSES
annual=con.execute("""
SELECT year, COUNT(*) crashes, SUM(COALESCE(injured,0)) persons_injured,
 SUM(COALESCE(killed,0)) persons_killed
FROM valid_crashes GROUP BY year ORDER BY year
""").df()
annual['yoy_pct']=annual['crashes'].pct_change().mul(100).round(2)
monthly=con.execute("""
SELECT month, COUNT(*) crashes, SUM(COALESCE(injured,0)) persons_injured,
 SUM(COALESCE(killed,0)) persons_killed
FROM valid_crashes GROUP BY month ORDER BY month
""").df()
borough=con.execute("""
SELECT borough, COUNT(*) crashes, SUM(COALESCE(injured,0)) persons_injured,
 SUM(COALESCE(killed,0)) persons_killed,
 ROUND(100.0*AVG(casualty_crash),2) casualty_crash_pct
FROM valid_crashes GROUP BY borough ORDER BY crashes DESC
""").df()
streets=con.execute("""
SELECT COALESCE(on_street,off_street) street, borough, COUNT(*) crashes,
 SUM(COALESCE(injured,0)) persons_injured, SUM(COALESCE(killed,0)) persons_killed
FROM valid_crashes WHERE COALESCE(on_street,off_street) IS NOT NULL
GROUP BY 1,2 ORDER BY crashes DESC LIMIT 10
""").df()
intersections=con.execute("""
SELECT borough, on_street, cross_street, COUNT(*) crashes,
 SUM(COALESCE(injured,0)) persons_injured
FROM valid_crashes WHERE on_street IS NOT NULL AND cross_street IS NOT NULL
GROUP BY 1,2,3 ORDER BY crashes DESC LIMIT 10
""").df()
factors=con.execute("""
SELECT factor, COUNT(*) mentions FROM crash_factors GROUP BY factor
ORDER BY mentions DESC LIMIT 25
""").df()
borough_factors=con.execute("""
SELECT borough, factor, COUNT(*) mentions,
 ROW_NUMBER() OVER(PARTITION BY borough ORDER BY COUNT(*) DESC) factor_rank
FROM crash_factors WHERE borough!='UNKNOWN'
GROUP BY borough,factor QUALIFY factor_rank<=5
ORDER BY borough,factor_rank
""").df()
print('Yearly collisions (counts, NOT population- or traffic-adjusted rates):');display(annual.tail(12))
print('Borough collision counts:');display(borough)
print('Top 10 streets:');display(streets)
print('Top 10 intersections:');display(intersections)
print('Top five recorded factor mentions:');display(factors.head(5))
print('Top factors by borough:');display(borough_factors.head(20))
for name,table in {'annual_trends':annual,'monthly_trends':monthly,'borough_summary':borough,'top_streets':streets,'top_intersections':intersections,'contributing_factors':factors,'borough_factors':borough_factors}.items():
    table.to_csv(OUT/f'nyc_{name}.csv', index=False)


In [ ]:
# CELL 5 — ROAD-USER INJURIES, FATALITIES, VEHICLE GROUPS, SQL VALIDATION
road_users=con.execute("""
SELECT borough,
 SUM(COALESCE(ped_inj,0)) pedestrian_injured, SUM(COALESCE(ped_kill,0)) pedestrian_killed,
 SUM(COALESCE(cyc_inj,0)) cyclist_injured, SUM(COALESCE(cyc_kill,0)) cyclist_killed,
 SUM(COALESCE(mot_inj,0)) motorist_injured, SUM(COALESCE(mot_kill,0)) motorist_killed
FROM valid_crashes GROUP BY borough ORDER BY borough
""").df()
annual_users=con.execute("""
SELECT year, SUM(COALESCE(ped_inj,0)) pedestrian_injured,
 SUM(COALESCE(ped_kill,0)) pedestrian_killed,
 SUM(COALESCE(cyc_inj,0)) cyclist_injured,
 SUM(COALESCE(cyc_kill,0)) cyclist_killed,
 SUM(COALESCE(mot_inj,0)) motorist_injured,
 SUM(COALESCE(mot_kill,0)) motorist_killed
FROM valid_crashes GROUP BY year ORDER BY year
""").df()
vehicles=con.execute("""
SELECT vehicle_group, COUNT(*) vehicle_mentions,
 ROUND(AVG(casualty_crash)*100,2) pct_mentions_in_casualty_crashes,
 COUNT(*) FILTER (WHERE killed>0) mentions_in_fatal_crashes
FROM crash_vehicles GROUP BY vehicle_group
ORDER BY vehicle_mentions DESC
""").df()
validation=con.execute("""
SELECT (SELECT COUNT(*) FROM crashes) AS source_rows,
       (SELECT COUNT(*) FROM valid_crashes) AS valid_date_rows,
       (SELECT COUNT(*) FROM crashes WHERE crash_date IS NULL) AS invalid_date_rows,
       (SELECT COALESCE(SUM(crashes),0) FROM (SELECT COUNT(*) crashes FROM valid_crashes GROUP BY year)) AS annual_rows,
       (SELECT COALESCE(SUM(crashes),0) FROM (SELECT COUNT(*) crashes FROM valid_crashes GROUP BY borough)) AS borough_rows
""").df().iloc[0]
assert validation['source_rows']==validation['valid_date_rows']+validation['invalid_date_rows']
assert validation['valid_date_rows']==validation['annual_rows']==validation['borough_rows']
assert (annual[['persons_injured','persons_killed']].fillna(0)>=0).all().all()
print('ROAD USER TOTALS (counts, not per-exposure risk):');display(road_users)
print('VEHICLE ASSOCIATIONS (vehicle mentions, not causal attributions):');display(vehicles.head(15))
print('VALIDATION: PASS — date, year, borough counts reconcile')
for name,t in {'road_user_by_borough':road_users,'road_user_by_year':annual_users,'vehicle_type_summary':vehicles}.items():
    t.to_csv(OUT/f'nyc_{name}.csv',index=False)


In [ ]:
# CELL 6 — PUBLIC-SAFETY EXECUTIVE DASHBOARD
fig,ax=plt.subplots(2,2,figsize=(15,10))
if len(annual):
    ax[0,0].plot(annual['year'],annual['crashes'],marker='o')
    ax[0,0].set_title('Collision Records by Year')
    ax[0,0].set_xlabel('Year');ax[0,0].set_ylabel('Crashes')
b=borough[borough.borough!='UNKNOWN'].head(7)
if len(b):
    ax[0,1].barh(b.borough[::-1],b.crashes[::-1]);ax[0,1].set_title('Crashes by Recorded Borough')
if len(factors):
    f=factors.head(7).iloc[::-1]
    ax[1,0].barh(f.factor.str.slice(0,35),f.mentions)
    ax[1,0].set_title('Most-Reported Contributing Factors')
if len(annual_users):
    for col,label in [('pedestrian_injured','Pedestrians'),('cyclist_injured','Cyclists'),('motorist_injured','Motorists')]:
        ax[1,1].plot(annual_users.year,annual_users[col],label=label)
    ax[1,1].legend();ax[1,1].set_title('Injuries by Road-User Group')
for a in ax.flat: a.grid(alpha=.15);a.set_axisbelow(True)
fig.suptitle('NYC Motor Vehicle Collisions | Descriptive Safety Dashboard',fontsize=15)
fig.tight_layout(rect=(0,0,1,.97))
fig.savefig(OUT/'nyc_traffic_safety_dashboard.png',dpi=160,bbox_inches='tight')
plt.show()
print('Dashboard saved.')


In [ ]:
# CELL 7 — INSIGHT SUMMARY, RECOMMENDATIONS, REPRODUCIBILITY, AND ZIP
from zipfile import ZipFile, ZIP_DEFLATED

def mdtable(frame,columns,n=10):
    if frame is None or frame.empty: return 'Not available from the supplied dataset.'
    t=frame.loc[:,columns].head(n).copy()
    for c in t:
        t[c]=t[c].map(lambda x: '—' if pd.isna(x) else (f'{x:.2f}' if isinstance(x,(float,np.floating)) else str(x)))
    return '| '+' | '.join(columns)+' |\n| '+' | '.join(['---']*len(columns))+' |\n'+'\n'.join('| '+' | '.join(row)+' |' for row in t.itertuples(index=False,name=None))

# Note: first/last date range may be partial or dataset extraction may be incomplete.
startdate=str(audit.iloc[0]['earliest_date'])[:10]
enddate=str(audit.iloc[0]['latest_date'])[:10]
valid_n=int(validation['valid_date_rows'])
missing_borough=int(audit.iloc[0]['missing_borough'])
missing_coords=int(audit.iloc[0]['missing_coordinates'])
noted_factor=factors.iloc[0]['factor'] if len(factors) else 'Not available'
top_boro=borough[borough.borough!='UNKNOWN'].iloc[0]['borough'] if any(borough.borough!='UNKNOWN') else 'Not available'

recs=f'''# NYC Traffic Safety — Policy Recommendations

## Purpose and interpretation
These proposals use observed collision counts, recorded contributing-factor mentions, and injury/fatality totals. They **do not estimate per-resident or per-trip crash risk** without external exposure denominators. Recorded factors are not proof of cause.

## 1. Prioritize high-crash street corridors (High)
**Evidence:** Top street and intersection tables identify frequently recorded crash locations.
**Action:** Audit recurring corridors, sightlines, crossing times, intersection geometry, and turning conflicts before selecting street redesigns.
**Measure:** Verified collisions and injuries before/after interventions, ideally with traffic-volume adjustments.

## 2. Target speeding interventions after local verification (High)
**Evidence:** Borough-specific factor summaries allow screening for speeding-related reports.
**Action:** Match speeding mentions to crash locations and validated speed observations; consider targeted enforcement and safer street design.
**Measure:** Speed compliance, severe-injury reductions, and equity of enforcement.

## 3. Improve pedestrian crossing safety (High)
**Evidence:** Pedestrian injured/killed totals are broken down by borough and year.
**Action:** Prioritize dangerous crossing locations using on-site audits; consider shorter crossing distances and better signals.
**Measure:** Pedestrian injuries and fatalities; accessible crossing performance.

## 4. Strengthen cyclist safety at conflict hotspots (High)
**Evidence:** Cyclist injury counts can identify boroughs for deeper location audits.
**Action:** Assess protected bike lanes, junction protections, and loading conflict areas.
**Measure:** Cyclist injury counts plus exposure-adjusted rates when ridership data is available.

## 5. Address driver attention issues (Medium)
**Evidence:** The highest-frequency recorded contributing factor is **{noted_factor}**, subject to coding and reporting variation.
**Action:** Combine education, targeted operations and engineering changes, prioritizing verified recurring locations.
**Measure:** Factor-reporting changes, observed behavior and injury outcomes.

## 6. Evaluate vehicle involvement, not presumed fault (Medium)
**Evidence:** Vehicle groups can be ranked by crash *mentions* and association with casualty crashes.
**Action:** Review heavy-vehicle turning conflicts and loading practices where relevant.
**Measure:** Incident trends and proportion involving injury, with exposure denominators where available.
**Caution:** Multiple vehicles per crash mean vehicle associations are not independent collision counts.

## 7. Strengthen open-data reporting quality (Critical)
**Evidence:** **{missing_borough:,}** records lack a recognized borough and **{missing_coords:,}** records lack one or both coordinates.
**Action:** Improve geographic completeness, check repeated collision IDs, standardize vehicle labels, and document extract coverage.
**Measure:** Monthly completeness scorecard, collision-ID uniqueness and coding consistency.

## Implementation roadmap
- **0–30 days:** Confirm reporting-period completeness, validate hotspots and missing-location records.
- **30–90 days:** Field-review targeted corridors and design priority interventions.
- **Quarterly:** Compare injury outcomes and publish an updated quality scorecard.

## Limitations
Counts are not exposure-adjusted risk or causal estimates. Boundary-year totals may be incomplete; contributing factors and vehicle types are mentions, not unique crashes. Fatalities are rare and should not be ranked solely using small numbers.
'''
(OUT/'RECOMMENDATIONS.md').write_text(recs,encoding='utf-8')

readme=f'''# NYC Traffic Safety — SQL Collision Intelligence

![NYC Traffic Safety Executive Dashboard](nyc_traffic_safety_dashboard.png)

## Business objective
Support NYC road-safety planning with reproducible SQL analysis of crashes, recorded factors, affected road-user groups and locations.

## Executive overview
| Measure | Value |
|---|---:|
| Original source rows | {int(validation['source_rows']):,} |
| Records with valid dates | {valid_n:,} |
| Invalid/missing crash dates | {int(validation['invalid_date_rows']):,} |
| Missing/unknown borough | {missing_borough:,} |
| Missing latitude or longitude | {missing_coords:,} |
| Source date coverage | {startdate} to {enddate} |
| Most collisions among known boroughs | {top_boro} |
| Most reported contributing factor | {noted_factor} |

## Annual collision counts
{mdtable(annual,['year','crashes','yoy_pct'],12)}

**Critical trend caveat:** Dataset coverage begins on {startdate} and ends on {enddate}; the first and last years may be partial. Year-over-year differences are descriptive and may reflect reporting completeness or changes. Do not conclude that crashes are rising/falling from unmatched partial years.

## Borough breakdown
{mdtable(borough,['borough','crashes','persons_injured','persons_killed'],10)}

The counts are **not borough crash rates**. Population, vehicles, trips, or miles traveled are needed for exposure-adjusted comparisons.

## Top streets
{mdtable(streets,['street','borough','crashes'],10)}

## Top intersections
{mdtable(intersections,['borough','on_street','cross_street','crashes'],10)}

Street aliases, direction and intersection ordering may split equivalent locations. Check geospatial quality before operational decisions.

## Top contributing factors
{mdtable(factors,['factor','mentions'],10)}

Each crash can contain as many as five contributing-factor fields. These totals represent **mentions**, not mutually exclusive collisions or verified causes. Borough comparisons are available in `nyc_borough_factors.csv`.

## Road-user injury and fatality analysis
{mdtable(road_users,['borough','pedestrian_injured','pedestrian_killed','cyclist_injured','cyclist_killed','motorist_injured','motorist_killed'],10)}

These are affected persons in reported crashes, not probabilities of harm; subgroup risk requires exposure counts. Missing values were converted to zero for summing, and original field completeness is audited.

## Vehicle involvement
{mdtable(vehicles,['vehicle_group','vehicle_mentions','pct_mentions_in_casualty_crashes'],12)}

Vehicle mention counts are not crash counts. A crash may list multiple vehicle groups; a reported casualty crash cannot establish which vehicle caused the harm. Standardization combines examples such as SUVs and station wagons and can be refined.

## Method and SQL implementation
1. Load CSV as strings through DuckDB's `read_csv` SQL scanner to handle large data without loading all raw rows into pandas.
2. Map column variants and parse crash dates using explicit date formats.
3. Build standardized crash, factor and vehicle SQL views.
4. Audit missing geography, invalid dates and duplicate collision IDs.
5. Aggregate annual/monthly, borough, location, factor, injury and vehicle data in SQL.
6. Validate reconciliation of dated crash rows across aggregates.
7. Export report-ready CSVs, dashboard, recommendations and ZIP.

## Data source and access
Input: `{DATA_FILE.name}`. This file is **not automatically included in the ZIP or GitHub upload** because it may be very large. To reproduce, obtain the original NYPD Motor Vehicle Collisions — Crashes open-data extract and set `DATA_FILE` if the filename differs. An extract's date range and updates may affect results. Publish the raw file separately only when appropriate and consistent with its source terms.

## Technology
Python, DuckDB (SQL), pandas, NumPy, Matplotlib, Google Colab.

## Reproduce
1. Upload `NYC_Traffic_Safety_SQL_Analysis.ipynb` to Google Colab.
2. Upload the NYC collisions CSV to Colab's `/content` or store in Drive and set `DATA_FILE` to the full path.
3. Run cells top-to-bottom. Results will be saved in `/content/NYC_Project7_Outputs`.
4. Download the project ZIP from the final cell. Download `.ipynb` separately and include it in your GitHub repository.
5. On GitHub, create `NYC-Traffic-Safety-Collision-Analytics` at https://github.com/new and upload extracted exports, the notebook, README and recommendations.

## Included outputs
- `nyc_annual_trends.csv`, `nyc_monthly_trends.csv`
- `nyc_borough_summary.csv`, `nyc_top_streets.csv`, `nyc_top_intersections.csv`
- `nyc_contributing_factors.csv`, `nyc_borough_factors.csv`
- `nyc_road_user_by_borough.csv`, `nyc_road_user_by_year.csv`
- `nyc_vehicle_type_summary.csv`, `nyc_data_quality_audit.csv`, `nyc_duplicate_collision_ids_audit.csv`
- `nyc_traffic_safety_dashboard.png`, `RECOMMENDATIONS.md`, `requirements.txt`, `.gitignore`

## Scope limitations and next steps
The source does not establish exposure-adjusted borough or road-user risk. Check coverage at boundaries and duplicate collision IDs. Additional Power BI dashboard development would be a separate deliverable; this release provides a Python dashboard and SQL-backed exported summaries suitable for Power BI.
'''
(OUT/'README.md').write_text(readme,encoding='utf-8')
(OUT/'requirements.txt').write_text('duckdb>=1.1,<2\npandas>=2.0,<4\nnumpy>=1.24,<3\nmatplotlib>=3.7,<4\n',encoding='utf-8')
(OUT/'.gitignore').write_text('*.duckdb\n*.wal\n.ipynb_checkpoints/\n__pycache__/\n.DS_Store\n',encoding='utf-8')

# Source CSV and sizable SQL database deliberately excluded.
zipfile=ROOT/'NYC_Project7_Traffic_Safety_Portfolio.zip'
pack=[p for p in OUT.iterdir() if p.is_file() and p.suffix.lower() in ('.csv','.png','.md','.txt')]
pack += [OUT/'.gitignore']
pack=[p for p in pack if p.exists()]
required_output=['README.md','RECOMMENDATIONS.md','nyc_annual_trends.csv','nyc_borough_summary.csv','nyc_contributing_factors.csv','nyc_traffic_safety_dashboard.png']
assert all((OUT/n).exists() for n in required_output)
with ZipFile(zipfile,'w',ZIP_DEFLATED) as z:
    for p in pack: z.write(p,arcname=p.name)
with ZipFile(zipfile) as z:
    assert z.testzip() is None
    assert all(x in z.namelist() for x in required_output)
print('VALIDATION: PASS')
print('README: CREATED; RECOMMENDATIONS: CREATED')
print('ZIP INTEGRITY: PASS; FILES:',len(pack))
print('PACKAGE READY:',zipfile)
print('SOURCE DATA: not duplicated in ZIP; SQL DB: not included')
print('NOTEBOOK: Download .ipynb separately from Colab')
if Path('/content').exists():
    from google.colab import files
    files.download(str(zipfile))
